# Iterative Intelligence Playbook: The Path to Benchmark-Beating RL

This notebook illustrates the systematic, data-driven process we have followed to optimize the SAC trading agent. Each phase represents a strategic "pivot" based on experiment results and AI-led analysis.

## 🛠️ Phase 1: Solving the Inactivity Collapse

Initially, the SAC agent suffered from "inactivity collapse," where it would simply stay in cash (0 shares) to avoid transaction costs.

**The Pivot:**
- **Entropy Tuning (`ent_coef`)**: Decreased from `auto` to `0.01` to encourage more stable but directional exploration.
- **Action Bonus (`reward_action_bonus_scale`)**: Introduced a small nudge (0.02) to reward the agent for simply taking a position (Buy/Sell) rather than idling.

## 📊 Phase 2: Stationarity & News Integration

Direct OHLCV prices are non-stationary and often lead to models that simply "memorize" price levels rather than learning signals.

**The Pivot:**
- **Stationary Features**: Switched to log returns, relative ranges, and normalized volumes.
- **News Sentiment**: Integrated Gemini-processed news sentiment into the daily observation frame to provide "regime awareness."

In [ ]:
# Fix pathing for notebook-level imports
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Example: Generating the optimized training frame
from src.market_data import get_tech_training_data

df = get_tech_training_data(
    include_news=True, 
    use_stationary_features=True
)
print(f"Training columns: {df.columns.tolist() if 'df' in locals() else 'Run this cell to see'}")

## 📈 Phase 3: Advanced Technical Indicator Expansion

To give the agent deeper insight into market volatility and trend, we expanded the observation space beyond simple returns.

**The Expansion:**
- **ATR (Average True Range)**: For volatility scaling.
- **Bollinger Band Width**: For squeeze/expansion detection.
- **SMA Crossovers**: To provide a long-term directional anchor.
- **VWAP (Relative)**: Connects price action to volume distribution.
- **MACD Signal/Histogram**: Captures momentum acceleration.

## 🤖 Phase 4: AI Strategic Analyst Integration

Raw numbers in a CSV don't always tell the whole story. We integrated **Gemini 2.0 Flash** into our reporting pipeline to act as a "Senior Quant Critic."

**The Workflow:**
1. **Experiment** -> Generates `experiment_leaderboard.csv`
2. **Quant Report** -> Renders markdown stats.
3. **AI Critique** -> Provides a "Confidence Score" and "Strategic Pivot" recommendation.

## 🔄 The Iterative Tuning Loop

Tuning an RL agent is not a one-shot process; it is a **continuous feedback loop** between the researcher (you) and the critic (the AI Analyst).

1. **Hypothesis Formulation**: We propose a change, like "Adding Bollinger Bands will provide a volatility signal that reduces drawdown."
2. **The Sweep**: We run a multi-seeded experiment to ensure the signal is robust across different initializations.
3. **Statistical Diagnostic**: We use `quant_report.py` to check the math (Alpha vs QQQ, Sharpe, Win Rate).
4. **The LLM Critique**: The **AI Strategic Analyst** looks at the "personality" of the agent (is it too fearful/inactive? is it overconfident/overfitting?).
5. **Strategic Pivot**: We adjust the hyperparameters (the "knobs") based on the AI's critique and run again.

## 🚀 Current Frontier: The Goldilocks Zone

Our current `hybrid-v1` run focuses on finding the **"Goldilocks Zone""** between exploration, training time, and action confidence.

**Current Hyperparameters (`hybrid-v1`):**
- `ent_coef`: 0.05 (Balanced exploration)
- `reward_action_bonus_scale`: 0.10 (Strong nudge to trade)
- `timesteps`: 40,000 (Allowing for policy stabilization)
- `reward_mode`: `sharpe` (Switching to full risk-adjusted metrics)

---